# Notebook 2: Baseline Logistic Regression

Two baselines for the heart-disease task:
1. **sklearn `LogisticRegression`** with L-BFGS — quasi-Newton reference solver.
2. **Hand-rolled full-batch gradient descent** — same loss, written out so the paper can show the explicit update rule.

Both predict probabilities (ROC-AUC is the competition metric).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, time, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SPLITS_DIR = '/content/drive/MyDrive/ECE567_Final/splits'
SEED = 2540
np.random.seed(SEED)

### Load the raw split saved by notebook 01

In [ ]:
raw = np.load(os.path.join(SPLITS_DIR, 'split_raw.npz'), allow_pickle=True)
X_tr_raw, X_val_raw = raw['X_tr'], raw['X_val']
y_tr, y_val = raw['y_tr'], raw['y_val']
feature_cols = list(raw['feature_cols'])

test_npz = np.load(os.path.join(SPLITS_DIR, 'test_raw.npz'), allow_pickle=True)
X_test_raw = test_npz['X_test']
test_ids   = test_npz['test_ids']

print('train:', X_tr_raw.shape, 'val:', X_val_raw.shape, 'test:', X_test_raw.shape)
print('features:', feature_cols)

### Preprocess: standardize continuous, one-hot nominal categoricals

From the EDA we know:
- **Continuous**: Age, BP, Cholesterol, Max HR, ST depression — standardize
- **Nominal categorical**: Chest pain type (1-4), EKG results (0-2), Thallium (3/6/7) — one-hot
- **Ordinal**: Slope of ST (1-3), Number of vessels fluro (0-3) — keep as ordered ints (standardize for scale)
- **Binary**: Sex, FBS over 120, Exercise angina — leave as 0/1

Fit transformers on train only; apply to val and test.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

continuous = ['Age', 'BP', 'Cholesterol', 'Max HR', 'ST depression',
              'Slope of ST', 'Number of vessels fluro']
nominal    = ['Chest pain type', 'EKG results', 'Thallium']
binary     = ['Sex', 'FBS over 120', 'Exercise angina']

col_idx = {c: feature_cols.index(c) for c in feature_cols}
cont_ix = [col_idx[c] for c in continuous]
nom_ix  = [col_idx[c] for c in nominal]
bin_ix  = [col_idx[c] for c in binary]

preproc = ColumnTransformer([
    ('cont', StandardScaler(), cont_ix),
    ('nom',  OneHotEncoder(drop='first', sparse_output=False), nom_ix),
    ('bin',  'passthrough', bin_ix),
])

X_tr  = preproc.fit_transform(X_tr_raw).astype(np.float64)
X_val = preproc.transform(X_val_raw).astype(np.float64)
X_test = preproc.transform(X_test_raw).astype(np.float64)

ohe_names = preproc.named_transformers_['nom'].get_feature_names_out(nominal).tolist()
feature_names_expanded = continuous + ohe_names + binary

print('post-preproc train shape:', X_tr.shape)
print('feature names (', len(feature_names_expanded), '):')
for n in feature_names_expanded:
    print(' ', n)

In [ ]:
# persist preprocessed arrays so downstream notebooks don't redo it
np.savez(os.path.join(SPLITS_DIR, 'split_preproc.npz'),
         X_tr=X_tr, X_val=X_val, X_test=X_test,
         y_tr=y_tr, y_val=y_val,
         test_ids=test_ids,
         feature_names=np.array(feature_names_expanded))
print('saved split_preproc.npz')

## Baseline 1 — sklearn LogisticRegression (L-BFGS)

Use a very large `C` to approximate an unregularized fit. L-BFGS is a quasi-Newton method, so this is effectively the reference "best convex optimum" for this loss.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, log_loss

t0 = time.time()
sk_lr = LogisticRegression(
    penalty='l2', C=1e8, solver='lbfgs', max_iter=2000, n_jobs=-1,
)
sk_lr.fit(X_tr, y_tr)
sk_train_time = time.time() - t0

sk_p_tr  = sk_lr.predict_proba(X_tr)[:, 1]
sk_p_val = sk_lr.predict_proba(X_val)[:, 1]

print(f'sklearn LR  train time: {sk_train_time:.2f}s')
print(f'sklearn LR  train AUC : {roc_auc_score(y_tr, sk_p_tr):.5f}')
print(f'sklearn LR  val   AUC : {roc_auc_score(y_val, sk_p_val):.5f}')
print(f'sklearn LR  val   logloss: {log_loss(y_val, sk_p_val):.5f}')

## Baseline 2 — Hand-rolled vanilla full-batch gradient descent

Logistic regression with binary cross-entropy loss:
$$
\mathcal{L}(\mathbf{w}, b) = -\frac{1}{N}\sum_{i=1}^{N}\bigl[\,y_i\log\sigma(z_i) + (1-y_i)\log(1-\sigma(z_i))\bigr], \quad z_i = \mathbf{w}^\top \mathbf{x}_i + b.
$$
Gradient:
$$
\nabla_{\mathbf{w}}\mathcal{L} = \frac{1}{N}\mathbf{X}^\top(\sigma(\mathbf{Xw}+b)-\mathbf{y}), \quad \partial_b\mathcal{L} = \frac{1}{N}\sum_i (\sigma(z_i)-y_i).
$$
Update with step size $\eta$:
$$
\mathbf{w} \leftarrow \mathbf{w} - \eta \nabla_{\mathbf{w}}\mathcal{L}, \qquad b \leftarrow b - \eta\,\partial_b\mathcal{L}.
$$

In [ ]:
def sigmoid(z):
    # numerically stable
    out = np.empty_like(z)
    pos = z >= 0
    out[pos] = 1.0 / (1.0 + np.exp(-z[pos]))
    ez = np.exp(z[~pos])
    out[~pos] = ez / (1.0 + ez)
    return out

def bce_loss(y, p, eps=1e-12):
    p = np.clip(p, eps, 1 - eps)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

def fit_gd(X, y, lr=0.1, n_iter=500, log_every=10, X_val=None, y_val=None):
    N, D = X.shape
    w = np.zeros(D)
    b = 0.0
    history = {'iter': [], 'train_loss': [], 'val_auc': []}
    for k in range(n_iter):
        z = X @ w + b
        p = sigmoid(z)
        grad_w = (X.T @ (p - y)) / N
        grad_b = (p - y).mean()
        w -= lr * grad_w
        b -= lr * grad_b
        if k % log_every == 0 or k == n_iter - 1:
            history['iter'].append(k)
            history['train_loss'].append(bce_loss(y, p))
            if X_val is not None:
                history['val_auc'].append(roc_auc_score(y_val, sigmoid(X_val @ w + b)))
    return w, b, history

In [ ]:
t0 = time.time()
w_gd, b_gd, hist_gd = fit_gd(X_tr, y_tr, lr=0.5, n_iter=500, log_every=10,
                              X_val=X_val, y_val=y_val)
gd_train_time = time.time() - t0

p_val_gd = sigmoid(X_val @ w_gd + b_gd)
p_tr_gd  = sigmoid(X_tr  @ w_gd + b_gd)

print(f'vanilla GD  train time: {gd_train_time:.2f}s ({len(hist_gd["iter"])} logs over 500 iters)')
print(f'vanilla GD  train AUC : {roc_auc_score(y_tr, p_tr_gd):.5f}')
print(f'vanilla GD  val   AUC : {roc_auc_score(y_val, p_val_gd):.5f}')
print(f'vanilla GD  val   logloss: {log_loss(y_val, p_val_gd):.5f}')

### Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(hist_gd['iter'], hist_gd['train_loss'])
axes[0].set_xlabel('iteration'); axes[0].set_ylabel('train BCE')
axes[0].set_title('Vanilla GD: training loss')
axes[0].grid(alpha=0.3)

axes[1].plot(hist_gd['iter'], hist_gd['val_auc'])
axes[1].axhline(roc_auc_score(y_val, sk_p_val), ls='--', color='gray',
                label=f'sklearn L-BFGS = {roc_auc_score(y_val, sk_p_val):.4f}')
axes[1].set_xlabel('iteration'); axes[1].set_ylabel('val AUC')
axes[1].set_title('Vanilla GD: validation AUC')
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/ECE567_Final/splits/fig_baseline_gd_curves.png',
            dpi=150, bbox_inches='tight')
plt.show()

### Coefficient comparison (sklearn vs hand-rolled GD)

If the hand-rolled solver converged, the two coefficient vectors should be very close.

In [ ]:
coefs = pd.DataFrame({
    'feature'    : feature_names_expanded,
    'sklearn'    : sk_lr.coef_.ravel(),
    'vanilla_gd' : w_gd,
})
coefs['abs_diff'] = (coefs['sklearn'] - coefs['vanilla_gd']).abs()
print(coefs.to_string(index=False))
print(f'\nbias sklearn = {sk_lr.intercept_[0]:.4f}   |   bias gd = {b_gd:.4f}')
print(f'max |sklearn - gd| coefficient diff: {coefs["abs_diff"].max():.4f}')

### Save baseline results for the paper

In [ ]:
results = {
    'sklearn_lbfgs': {
        'train_time_s': sk_train_time,
        'train_auc'   : float(roc_auc_score(y_tr,  sk_p_tr)),
        'val_auc'     : float(roc_auc_score(y_val, sk_p_val)),
        'val_logloss' : float(log_loss(y_val, sk_p_val)),
        'n_iter'      : int(sk_lr.n_iter_[0]),
    },
    'vanilla_gd': {
        'train_time_s': gd_train_time,
        'train_auc'   : float(roc_auc_score(y_tr,  p_tr_gd)),
        'val_auc'     : float(roc_auc_score(y_val, p_val_gd)),
        'val_logloss' : float(log_loss(y_val, p_val_gd)),
        'n_iter'      : 500,
        'lr'          : 0.5,
    },
}
with open(os.path.join(SPLITS_DIR, 'results_baselines.json'), 'w') as fh:
    json.dump(results, fh, indent=2)
print(json.dumps(results, indent=2))